In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "bladder"
cell_state = None

reference = "GRCh38"
paper_doi = "https://doi.org/10.1681/ASN.2019040335",
table_link = "https://cdn-links.lww.com/permalink/jsn/c/jsn_30_11_2022_12_07_yu_2019040335_sdc5.xlsx"

# don't include in header
table_name = "jsn_30_11_2022_12_07_yu_2019040335_sdc5.xlsx"
table_name = "clean_degs.xlsx" 

## Read in file

In [5]:
excel = pd.read_excel(table_name, skiprows = 1)

df = excel

In [6]:
df.head()

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
0,0.0,0.801753,0.994,0.625,0.0,basal cells 1,Igfbp2
1,0.0,0.765484,0.846,0.429,0.0,basal cells 1,Gsdmc2
2,0.0,0.657349,0.995,0.732,0.0,basal cells 1,Krt15
3,0.0,0.625553,0.928,0.532,0.0,basal cells 1,Acaa1b
4,0.0,0.603922,0.949,0.549,0.0,basal cells 1,Krt5


## Perform the filtering

In [7]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [8]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [9]:
markers

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
5644,0.000000e+00,2.574907,0.377,0.001,0.000000e+00,T cells,Nkg7
5642,0.000000e+00,4.359741,0.377,0.006,0.000000e+00,T cells,Ccl5
5647,0.000000e+00,2.298859,0.554,0.000,0.000000e+00,T cells,Cd3g
5646,0.000000e+00,2.347063,0.631,0.001,0.000000e+00,T cells,Trbc2
5720,3.590000e-84,2.353474,0.654,0.130,5.620000e-80,T cells,AW112010
...,...,...,...,...,...,...,...
2736,0.000000e+00,2.211725,1.000,0.361,0.000000e+00,myofibroblast,Col1a2
2735,0.000000e+00,2.300105,1.000,0.399,0.000000e+00,myofibroblast,Sparc
2734,0.000000e+00,2.396431,1.000,0.314,0.000000e+00,myofibroblast,Col1a1
2731,0.000000e+00,3.012099,1.000,0.080,0.000000e+00,myofibroblast,Car3


In [10]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
T cells                0.585857
smooth muscle cells    0.747800
dendritic cells        0.853833
fibroblast 3           0.871667
monocytes              0.889200
neurone                0.895714
endothelial cells      0.920571
fibroblast 2           0.988000
myofibroblast          0.992000
Name: pct.1, dtype: float64

In [11]:
markers[col_ct].value_counts()

cluster
monocytes              20
endothelial cells      14
smooth muscle cells    10
T cells                 7
neurone                 7
dendritic cells         6
myofibroblast           6
fibroblast 3            3
fibroblast 2            2
Name: count, dtype: int64

## Find Ensembl ID


In [12]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [13]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [14]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [15]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [16]:
save

[{'cell_type_label': 'T cells',
  'gene': 'Nkg7',
  'organism': 'homo_sapiens',
  'cell_source': 'bladder',
  'cell_state': None,
  'gene_id': 'ENSG00000105374'},
 {'cell_type_label': 'T cells',
  'gene': 'Ccl5',
  'organism': 'homo_sapiens',
  'cell_source': 'bladder',
  'cell_state': None,
  'gene_id': 'ENSG00000271503'},
 {'cell_type_label': 'T cells',
  'gene': 'Cd3g',
  'organism': 'homo_sapiens',
  'cell_source': 'bladder',
  'cell_state': None,
  'gene_id': 'ENSG00000289746'},
 {'cell_type_label': 'T cells',
  'gene': 'Trbc2',
  'organism': 'homo_sapiens',
  'cell_source': 'bladder',
  'cell_state': None,
  'gene_id': 'ENSG00000211772'},
 {'cell_type_label': 'T cells',
  'gene': 'AW112010',
  'organism': 'homo_sapiens',
  'cell_source': 'bladder',
  'cell_state': None,
  'gene_id': 'gene not found'},
 {'cell_type_label': 'dendritic cells',
  'gene': 'Cd209a',
  'organism': 'homo_sapiens',
  'cell_source': 'bladder',
  'cell_state': None,
  'gene_id': 'gene not found'},
 {'cell_t

## Save evidence

In [17]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 